# Tarea 1 - Clase 8: Autogestión NoSQL

## Instrucciones
En esta tarea pondrán en práctica el manejo de Bases de Datos No-Relacionales mediante archivos planos (CSV). 

A diferencia de SQL, aquí no contamos con un motor que actualice filas específicas; deberán aplicar la lógica de leer todo el archivo en la memoria RAM, modificar los datos con Python y sobrescribir el archivo completo.

## Instrucciones de Autogestión:

Tomando como base el código de la clase Cliente visto en el Live Coding, deben implementar las 4 operaciones CRUD para el módulo de Proveedor.

1. Clase Base: Asegúrense de tener definida la clase Proveedor con sus atributos básicos (NIT, Nombre de la Empresa, Ciudad).

2. Create: Escriban una función que reciba un objeto Proveedor y lo guarde como una nueva fila en un archivo llamado proveedores_nosql.csv.

3. Read: Escriban una función que lea el archivo CSV e imprima el listado de proveedores en la consola.

4. Update: Escriban una función que busque a un proveedor por su NIT, actualice su ciudad en la memoria y sobrescriba el archivo CSV.

5. Delete: Escriban una función que elimine a un proveedor específico (por su NIT) reescribiendo el archivo sin esa fila.

## Requisitos de Entrega:

Formato: Archivo .ipynb con el código funcional y las pruebas de ejecución de cada función.

Fecha Límite: Miércoles, 15 de abril de 2026, antes de las 16:00 PM (Previo al Segundo Parcial).

“Entender cómo se leen y escriben los archivos planos es el primer paso para dominar el procesamiento masivo de datos.”

In [1]:
# ==============================================================================
# UNIVERSIDAD DE LA SABANA - EICEA
# SOLUCIÓN IDEAL: TAREA 1 - CLASE 8 (AUTOGESTIÓN NOSQL)
# TEMA: Operaciones CRUD en Archivos Planos (CSV)
# ==============================================================================

# Importamos el módulo 'csv' estándar de Python para manejar archivos separados por comas
import csv
# Importamos el módulo 'os' para interactuar con el sistema operativo (ej. verificar si un archivo existe)
import os

# Definimos el nombre del archivo plano que actuará como nuestra Base de Datos NoSQL
archivo_csv = "tarea_1_proveedores_nosql.csv"

# ==============================================================================
# 1. CLASE BASE: PROVEEDOR
# ==============================================================================
class Proveedor:
    """Clase que representa la entidad Proveedor en nuestro sistema."""
    
    # Constructor de la clase que inicializa los atributos básicos
    def __init__(self, nit, nombre_empresa, ciudad):
        # Convertimos el NIT a string para evitar problemas de formato al leer/escribir en texto plano
        self.nit = str(nit)
        self.nombre_empresa = nombre_empresa
        self.ciudad = ciudad


# ==============================================================================
# 2. OPERACIONES CRUD (Create, Read, Update, Delete)
# ==============================================================================

# --- CREATE (Crear e Insertar) ---
def crear_proveedor_csv(proveedor):
    """Recibe un objeto Proveedor y lo guarda como una nueva fila en el archivo CSV."""
    
    # Verificamos si el archivo ya existe en la ruta actual. Devuelve True o False.
    archivo_existe = os.path.isfile(archivo_csv)
    
    # Abrimos el archivo en modo 'a' (append/adjuntar). Esto agrega datos al final sin borrar lo anterior.
    # 'newline=""' es obligatorio en Windows para evitar que se inserten filas en blanco entre registros.
    # 'encoding="utf-8"' asegura que caracteres como tildes o la 'ñ' se guarden correctamente.
    with open(archivo_csv, mode='a', newline='', encoding='utf-8') as file:
        
        # Creamos el objeto escritor de CSV asociado a nuestro archivo
        writer = csv.writer(file)
        
        # Si el archivo es nuevo (no existía), escribimos primero la fila de encabezados
        if not archivo_existe:
            writer.writerow(["NIT", "Nombre Empresa", "Ciudad"])
            
        # Escribimos los atributos del objeto proveedor como una lista (una nueva fila en el CSV)
        writer.writerow([proveedor.nit, proveedor.nombre_empresa, proveedor.ciudad])
        
    print(f"✅ Proveedor '{proveedor.nombre_empresa}' guardado exitosamente en el CSV.")


# --- READ (Leer y Mostrar) ---
def leer_proveedores_csv():
    """Lee el archivo CSV completo e imprime el listado de proveedores en la consola."""
    
    print("\n🏢 BASE DE DATOS PROVEEDORES (NoSQL - Archivo CSV):")
    
    # Sanity Check: Verificamos si el archivo existe antes de intentar leerlo para evitar errores
    if not os.path.isfile(archivo_csv):
        print("⚠️ El archivo aún no existe. Registre un proveedor primero.")
        return # Cortamos la ejecución de la función aquí
        
    # Abrimos el archivo en modo 'r' (read/lectura)
    with open(archivo_csv, mode='r', encoding='utf-8') as file:
        
        # Creamos el objeto lector de CSV
        reader = csv.reader(file)
        
        # Iteramos línea por línea sobre el contenido del archivo
        for fila in reader:
            # Unimos los elementos de la lista 'fila' usando " | " como separador visual y lo imprimimos
            print(" | ".join(fila))
            
    print("-" * 60)


# --- UPDATE (Actualizar) ---
def actualizar_ciudad_proveedor(nit_buscar, nueva_ciudad):
    """Busca un proveedor por su NIT, actualiza su ciudad en memoria y sobrescribe el archivo."""
    
    # Lista temporal en la memoria RAM para guardar todas las filas del archivo
    filas_temporales = []
    # Bandera lógica para saber si encontramos y modificamos al proveedor
    actualizado = False
    
    # PASO A: Leer todo el archivo y cargarlo a la memoria RAM
    with open(archivo_csv, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        
        for fila in reader:
            # Comparamos el NIT de la fila actual (índice 0) con el NIT que estamos buscando
            if fila[0] == str(nit_buscar):
                # Si coincide, modificamos la ciudad (índice 2) en la lista de esta fila
                fila[2] = nueva_ciudad
                actualizado = True # Marcamos que la actualización fue exitosa
                
            # Guardamos la fila (modificada o no) en nuestra lista temporal de la RAM
            filas_temporales.append(fila)
            
    # PASO B: Sobrescribir todo el archivo desde cero
    if actualizado:
        # Abrimos en modo 'w' (write/escribir). ¡CUIDADO! Esto borra todo el contenido anterior del archivo.
        with open(archivo_csv, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            # Escribimos toda la matriz de datos (lista de listas) que teníamos en la RAM de una sola vez
            writer.writerows(filas_temporales)
            
        print(f"🔄 Proveedor con NIT {nit_buscar} actualizado. Nueva ciudad: {nueva_ciudad}.")
    else:
        # Si la bandera sigue en False, significa que el NIT no estaba en el archivo
        print(f"⚠️ Proveedor con NIT {nit_buscar} no fue encontrado en la base de datos.")


# --- DELETE (Eliminar) ---
def eliminar_proveedor_csv(nit_eliminar):
    """Elimina a un proveedor específico reescribiendo el archivo sin esa fila."""
    
    # Lista temporal en la memoria RAM
    filas_temporales = []
    # Bandera lógica para confirmar si el proveedor fue omitido (eliminado)
    eliminado = False
    
    # PASO A: Leer y filtrar en la memoria RAM
    with open(archivo_csv, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        
        for fila in reader:
            # Si el NIT de la fila NO ES el que queremos eliminar, lo guardamos en la memoria
            if fila[0] != str(nit_eliminar):
                filas_temporales.append(fila)
            else:
                # Si el NIT coincide, NO lo agregamos a la lista temporal (lo omitimos)
                eliminado = True
                
    # PASO B: Sobrescribir el archivo con los datos filtrados
    if eliminado:
        # Abrimos en modo 'w' (write) para reemplazar el archivo viejo con la nueva lista filtrada
        with open(archivo_csv, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerows(filas_temporales)
            
        print(f"❌ Proveedor con NIT {nit_eliminar} eliminado del archivo CSV.")
    else:
        print(f"⚠️ Proveedor con NIT {nit_eliminar} no fue encontrado en la base de datos.")


# ==============================================================================
# 3. PRUEBAS DE EJECUCIÓN (Sanity Check)
# ==============================================================================

# Este bloque solo se ejecuta si corremos este script directamente
if __name__ == "__main__":
    
    print("🚀 INICIANDO PRUEBAS CRUD NOSQL (CSV)...\n")
    
    # Limpiamos el archivo de pruebas anteriores si existe (para tener un entorno limpio)
    if os.path.isfile(archivo_csv):
        os.remove(archivo_csv)
        
    # 1. Instanciamos objetos de la clase Proveedor
    prov1 = Proveedor(nit="900111222", nombre_empresa="Distribuidora Sabana", ciudad="Bogotá")
    prov2 = Proveedor(nit="800333444", nombre_empresa="Insumos Panaderos S.A.", ciudad="Medellín")
    prov3 = Proveedor(nit="700555666", nombre_empresa="Cafetaleros de la Sierra", ciudad="Santa Marta")
    
    # 2. PRUEBA CREATE: Guardamos los proveedores en el CSV
    crear_proveedor_csv(prov1)
    crear_proveedor_csv(prov2)
    crear_proveedor_csv(prov3)
    
    # 3. PRUEBA READ: Leemos el archivo para verificar la inserción
    leer_proveedores_csv()
    
    # 4. PRUEBA UPDATE: Actualizamos la ciudad del proveedor 900111222 a "Chía"
    actualizar_ciudad_proveedor(nit_buscar="900111222", nueva_ciudad="Chía")
    leer_proveedores_csv()
    
    # 5. PRUEBA DELETE: Eliminamos al proveedor 800333444 (Insumos Panaderos S.A.)
    eliminar_proveedor_csv(nit_eliminar="800333444")
    leer_proveedores_csv()

🚀 INICIANDO PRUEBAS CRUD NOSQL (CSV)...

✅ Proveedor 'Distribuidora Sabana' guardado exitosamente en el CSV.
✅ Proveedor 'Insumos Panaderos S.A.' guardado exitosamente en el CSV.
✅ Proveedor 'Cafetaleros de la Sierra' guardado exitosamente en el CSV.

🏢 BASE DE DATOS PROVEEDORES (NoSQL - Archivo CSV):
NIT | Nombre Empresa | Ciudad
900111222 | Distribuidora Sabana | Bogotá
800333444 | Insumos Panaderos S.A. | Medellín
700555666 | Cafetaleros de la Sierra | Santa Marta
------------------------------------------------------------
🔄 Proveedor con NIT 900111222 actualizado. Nueva ciudad: Chía.

🏢 BASE DE DATOS PROVEEDORES (NoSQL - Archivo CSV):
NIT | Nombre Empresa | Ciudad
900111222 | Distribuidora Sabana | Chía
800333444 | Insumos Panaderos S.A. | Medellín
700555666 | Cafetaleros de la Sierra | Santa Marta
------------------------------------------------------------
❌ Proveedor con NIT 800333444 eliminado del archivo CSV.

🏢 BASE DE DATOS PROVEEDORES (NoSQL - Archivo CSV):
NIT | Nombre Em